In [ ]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [17]:

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):

  display(Javascript(RECORD))

  js_result = output.eval_js('record(%s)' % (sec * 1000))

  audio = b64decode(js_result.split(',')[1])

  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)

  return f'/content/{file_name}'

print('Ouvindo...\n')
record_file = record()

display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q

In [18]:
import whisper

language = "pt"

model = whisper.load_model("medium")

result = model.transcribe(record_file, fp16=False, language=language)

transcription = result["text"].strip()

print("Transcrição:", transcription)

Transcrição: Torre de Belém em Portugal


# 3. Integração com a API do ChatGPT 💬

In [2]:
!pip install groq

In [19]:
import os
from groq import Groq

# ⚠️ Coloque sua API KEY DA GROQ antes de executar o projeto
os.environ["GROQ_API_KEY"] = "COLOQUE_SUA_CHAVE_AQUI"

api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=api_key)

prompt = f"""
Você é um guia turístico experiente.
Explique de forma interessante e curta sobre o local mencionado pelo usuário.
Inclua curiosidades, história e dicas de visita.

Usuário disse: "{transcription}"
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

chatgpt_response = response.choices[0].message.content

print("Guia turístico diz:")
print(chatgpt_response)

Guia turístico diz:
A Torre de Belém! Um dos monumentos mais icônicos e misteriosos de Portugal.

Essa incrível estrutura está localizada em Lisboa, no coração da História de Portugal, e é considerada um Patrimônio Mundial da UNESCO. Aqui estão algumas curiosidades interessantes sobre a Torre de Belém:

**História**: Foi construída em 1515 pelo arquiteto militar Afonso da Guarda no século XV para ser um símbolo de poder e resistência contra ataques estrangeiros. Durante anos, a torre serviu como uma fortaleza militar.

**Curiosidades**:

* A Torre de Belém tem uma arquitetura única, com influências mouras e góticas.
* As pedras da torre foram transportadas por navios do Marrocos.
* A estrutura é protegida por uma muralha circular, que é uma das características mais distintas da torre.
* No interior da torre, você pode ver uma escultura de São Vicente, o padroeiro de Portugal, e uma estátua de D. Pedro de Meneses, um nobre português que luta contra os mouros.

**Dicas**:

* A melhores h

# 4. Sintetizando a Resposta do ChatGPT Como Voz (gTTS) 🔊

In [1]:
!pip install gTTS

In [20]:
from gtts import gTTS
from IPython.display import Audio, display

language = "pt"

gtts_object = gTTS(text=chatgpt_response, lang=language, slow=False)

response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

display(Audio(response_audio, autoplay=True))